In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **1. Document Collection**

In [ ]:
!pip install tqdm

In [ ]:
import os
import time
import random
import json
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
from typing import Dict, List, Set
from tqdm import tqdm


# =============================================================================
# 1. API QUERY
# =============================================================================

def get_response(query: str | None = None,
                 metadata: Dict[str, str] = {},
                 max_results: int = 100,
                 start: int = 0) -> str:
    """
    Query the arXiv API and return the raw XML response.

    Args:
        query:       Optional free-text search string.
        metadata:    Dict of arXiv field filters, e.g. {"cat": "cs.CL"}.
        max_results: Number of results to request per API call.
        start:       Pagination offset.

    Returns:
        Raw XML string from the arXiv API.
    """
    if query is not None:
        metadata['all'] = urllib.parse.quote(query)

    combined_query = '+AND+'.join([f'{k}:{v}' for k, v in metadata.items()])
    url = (
        f'http://export.arxiv.org/api/query?search_query={combined_query}'
        f'&start={start}&max_results={max_results}'
        f'&sortBy=lastUpdatedDate&sortOrder=descending'
    )

    max_retries = 5
    for attempt in range(max_retries):
        try:
            with urllib.request.urlopen(url, timeout=30) as response:
                return response.read().decode('utf-8')
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = (2 ** attempt) + random.uniform(0, 1)
                print(f"  [Retry {attempt + 1}/{max_retries}] API request failed. "
                      f"Waiting {wait_time:.1f}s... ({e})")
                time.sleep(wait_time)
            else:
                print(f"  [ERROR] Failed after {max_retries} attempts: {e}")
                raise


# =============================================================================
# 2. PDF DOWNLOADER
# =============================================================================

def download_pdfs(response: str,
                  download_dir: str,
                  existing_ids: Set[str],
                  target_count: int,
                  current_count: int) -> List[str]:
    """
    Parse an arXiv API XML response and download the listed PDFs.

    Args:
        response:      Raw XML string from arXiv API.
        download_dir:  Local folder to save PDFs into.
        existing_ids:  Paper IDs already on disk — skip these.
        target_count:  Total papers we want for this category.
        current_count: How many have already been downloaded.

    Returns:
        List of paper IDs successfully downloaded in this call.
    """
    os.makedirs(download_dir, exist_ok=True)
    downloaded_ids = []

    try:
        root = ET.fromstring(response)
        ns = {'atom': 'http://www.w3.org/2005/Atom'}

        for entry in root.findall('atom:entry', ns):
            if current_count + len(downloaded_ids) >= target_count:
                break

            try:
                # ── Extract paper ID ──────────────────────────────────────
                id_elem = entry.find('atom:id', ns)
                if id_elem is None or not id_elem.text:
                    print("  Skipping entry with no ID.")
                    continue

                paper_id = id_elem.text.split('/abs/')[-1]

                if paper_id in existing_ids or paper_id in downloaded_ids:
                    print(f"  Skipping duplicate: {paper_id}")
                    continue

                # ── Extract title & PDF link ──────────────────────────────
                title_elem = entry.find('atom:title', ns)
                title = title_elem.text.strip().replace('\n', ' ') if title_elem is not None else "Unknown"

                pdf_url = None
                for link in entry.findall('atom:link', ns):
                    if link.attrib.get('title') == 'pdf':
                        pdf_url = link.attrib['href']
                        break

                if not pdf_url:
                    print(f"  No PDF link found for: {title[:60]}")
                    continue

                # ── Download PDF ──────────────────────────────────────────
                file_path = os.path.join(download_dir, f"{paper_id}.pdf")
                if os.path.exists(file_path):
                    print(f"  Already on disk, skipping: {paper_id}")
                    continue

                print(f"  Downloading [{current_count + len(downloaded_ids) + 1}"
                      f"/{target_count}] {paper_id} — {title[:50]}...")

                max_retries = 5
                for attempt in range(max_retries):
                    try:
                        urllib.request.urlretrieve(pdf_url, file_path)
                        downloaded_ids.append(paper_id)
                        break
                    except Exception as e:
                        if attempt < max_retries - 1:
                            wait_time = (2 ** attempt) + random.uniform(0, 1)
                            print(f"    [Retry {attempt + 1}] Download failed. "
                                  f"Waiting {wait_time:.1f}s...")
                            time.sleep(wait_time)
                        else:
                            print(f"    [ERROR] Could not download {paper_id}: {e}")

                # Polite delay between downloads to avoid rate limiting
                time.sleep(random.uniform(0.5, 1.5))

            except Exception as e:
                print(f"  Error processing entry: {e}")
                continue

    except ET.ParseError as e:
        print(f"  [ERROR] XML parse error: {e}")
    except Exception as e:
        print(f"  [ERROR] Unexpected error in download_pdfs: {e}")

    return downloaded_ids


# =============================================================================
# 3. METADATA EXTRACTOR
# =============================================================================

def extract_arxiv_metadata(paper_id: str, output_folder: str) -> bool:
    """
    Fetch metadata for one arXiv paper via the API and save it as JSON.

    Args:
        paper_id:      arXiv paper ID string (e.g. "2402.01359v2").
        output_folder: Folder to save the JSON file into.

    Returns:
        True if metadata was saved successfully, False otherwise.
    """
    try:
        url = f'http://export.arxiv.org/api/query?id_list={paper_id}'
        namespaces = {
            'atom':       'http://www.w3.org/2005/Atom',
            'arxiv':      'http://arxiv.org/schemas/atom',
            'opensearch': 'http://a9.com/-/spec/opensearch/1.1/'
        }

        # ── Fetch ─────────────────────────────────────────────────────────
        max_retries = 5
        xml_data = None
        for attempt in range(max_retries):
            try:
                with urllib.request.urlopen(url, timeout=30) as response:
                    xml_data = response.read().decode('utf-8')
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    time.sleep(wait_time)
                else:
                    print(f"  [ERROR] Metadata fetch failed for {paper_id}: {e}")
                    return False

        # ── Parse ─────────────────────────────────────────────────────────
        root = ET.fromstring(xml_data)
        entry = root.find('.//atom:entry', namespaces)
        if entry is None:
            print(f"  No entry found for: {paper_id}")
            return False

        def text_of(tag):
            elem = entry.find(tag, namespaces)
            return elem.text.strip() if elem is not None and elem.text else ""

        authors = [
            author.find('./atom:name', namespaces).text
            for author in entry.findall('./atom:author', namespaces)
            if author.find('./atom:name', namespaces) is not None
        ]
        categories = [
            cat.get('term') for cat in entry.findall('./atom:category', namespaces)
            if cat.get('term')
        ]

        metadata = {
            'id':         paper_id,
            'title':      text_of('./atom:title'),
            'authors':    authors,
            'categories': categories,
            'abstract':   text_of('./atom:summary'),
            'updated':    text_of('./atom:updated'),
            'published':  text_of('./atom:published'),
        }

        # ── Save ──────────────────────────────────────────────────────────
        os.makedirs(output_folder, exist_ok=True)
        with open(os.path.join(output_folder, f"{paper_id}.json"), 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)

        return True

    except Exception as e:
        print(f"  [ERROR] extract_arxiv_metadata({paper_id}): {e}")
        return False


# =============================================================================
# 4. MAIN ORCHESTRATOR
# =============================================================================

def main(output_folder: str,
         existing_ids: Set[str],
         target_pdfs_per_category: int = 500) -> None:
    """
    Download target_pdfs_per_category PDFs from arXiv cs.CL and save metadata.

    Args:
        output_folder:            Root folder for all downloaded files.
        existing_ids:             Paper IDs already present — skip these.
        target_pdfs_per_category: Number of PDFs to download (default 500).
    """

    # ── Hardcoded to cs.CL — no YAML config file required ─────────────────
    arxiv_categories = {
        "Computation_and_Language": ["cs.CL"]
    }

    # ── Initialise download tracking ───────────────────────────────────────
    successful_downloads: Dict[str, List[str]] = {}
    for category in arxiv_categories:
        pdf_dir = os.path.join(output_folder, 'pdf', category)
        os.makedirs(pdf_dir, exist_ok=True)
        # Count PDFs already on disk so we can resume a partial run
        successful_downloads[category] = [
            name.replace('.pdf', '')
            for name in os.listdir(pdf_dir)
            if name.endswith('.pdf')
        ]
        already = len(successful_downloads[category])
        if already > 0:
            print(f"  Resuming: {already} PDFs already on disk for '{category}'.")

    # ── Download loop ──────────────────────────────────────────────────────
    for category in arxiv_categories:
        print(f"\n{'='*60}")
        print(f"  Category: {category}")
        print(f"  Target:   {target_pdfs_per_category} PDFs")
        print(f"{'='*60}")

        pdfs_downloaded = len(successful_downloads[category])
        results_offset  = 0
        batch_size      = 100
        pdf_dir         = os.path.join(output_folder, 'pdf', category)

        while pdfs_downloaded < target_pdfs_per_category:
            print(f"\n  Progress: {pdfs_downloaded}/{target_pdfs_per_category} "
                  f"| API offset: {results_offset}")

            for sub_category in arxiv_categories[category]:
                if pdfs_downloaded >= target_pdfs_per_category:
                    break

                # Date range: 1 year (20250101 - 20260101) to ensure 500+ cs.CL papers
                metadata = {
                    "cat":           sub_category,
                    "submittedDate": "[202501010600+TO+202601010600]"
                }

                try:
                    response = get_response(
                        metadata=metadata,
                        max_results=batch_size,
                        start=results_offset
                    )
                    new_ids = download_pdfs(
                        response=response,
                        download_dir=pdf_dir,
                        existing_ids=existing_ids,
                        target_count=target_pdfs_per_category,
                        current_count=pdfs_downloaded
                    )

                    pdfs_downloaded += len(new_ids)
                    successful_downloads[category].extend(new_ids)
                    existing_ids.update(new_ids)

                except Exception as e:
                    print(f"  [ERROR] {category} / {sub_category}: {e}")
                    continue

            results_offset += batch_size
            time.sleep(3)  # Polite pause between API page requests

            # Increased ceiling: 1000 → 5000 to accommodate 500 papers
            if results_offset > 5000:
                print(f"\n  [WARNING] Reached offset limit (5000) for {category}. "
                      f"Downloaded {pdfs_downloaded}/{target_pdfs_per_category}.")
                break

    # ── Metadata extraction ────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  Extracting metadata for all downloaded PDFs...")
    print(f"{'='*60}")

    metadata_folder = os.path.join(output_folder, 'metadata')
    os.makedirs(metadata_folder, exist_ok=True)

    total_success = 0
    total_failure = 0

    for category, paper_ids in successful_downloads.items():
        print(f"\n  [{category}] Processing {len(paper_ids)} papers...")
        cat_success = 0

        for paper_id in tqdm(paper_ids, desc=category):
            meta_path = os.path.join(metadata_folder, f"{paper_id}.json")
            if os.path.exists(meta_path):
                cat_success += 1
                continue
            if extract_arxiv_metadata(paper_id, metadata_folder):
                cat_success += 1
            else:
                total_failure += 1
            time.sleep(0.5)  # Polite delay between metadata requests

        total_success += cat_success
        print(f"  Metadata saved: {cat_success}/{len(paper_ids)}")

    # ── Final summary ──────────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print("  DOWNLOAD COMPLETE")
    print(f"{'='*60}")
    total_papers = 0
    for category, paper_ids in successful_downloads.items():
        print(f"  {category}: {len(paper_ids)} PDFs")
        total_papers += len(paper_ids)
    print(f"\n  Total PDFs:          {total_papers}")
    print(f"  Metadata successes:  {total_success}")
    print(f"  Metadata failures:   {total_failure}")
    print(f"\n  Output location:     {os.path.abspath(output_folder)}")
    print(f"{'='*60}\n")



# =============================================================================
# 5. CONFIG & ENTRY POINT
# =============================================================================
# Edit these two variables to change behaviour — no command-line args needed.
# This avoids the argparse conflict with Colab's kernel launcher arguments.

OUTPUT_FOLDER       = '/content/drive/MyDrive/arxiv_data'  # Where PDFs and metadata are saved
TARGET_PER_CATEGORY = 500                     # Number of cs.CL papers to download

# ── Run ───────────────────────────────────────────────────────────────────────
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Collect IDs already on disk so an interrupted run can be safely resumed
metadata_dir = os.path.join(OUTPUT_FOLDER, 'metadata')
os.makedirs(metadata_dir, exist_ok=True)
existing_ids: Set[str] = set(
    fname.replace('.json', '') for fname in os.listdir(metadata_dir)
    if fname.endswith('.json')
)
print(f"Found {len(existing_ids)} existing paper IDs - these will be skipped.")

main(OUTPUT_FOLDER, existing_ids, TARGET_PER_CATEGORY)

Found 500 existing paper IDs - these will be skipped.
  Resuming: 500 PDFs already on disk for 'Computation_and_Language'.

  Category: Computation_and_Language
  Target:   500 PDFs

  Extracting metadata for all downloaded PDFs...

  [Computation_and_Language] Processing 500 papers...


Computation_and_Language: 100%|██████████| 500/500 [00:00<00:00, 6232.75it/s]

  Metadata saved: 500/500

  DOWNLOAD COMPLETE
  Computation_and_Language: 500 PDFs

  Total PDFs:          500
  Metadata successes:  500
  Metadata failures:   0

  Output location:     /content/drive/MyDrive/arxiv_data



# **2. Document Processing**

In [ ]:
!pip install "mistralai==1.2.5" joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: eval-type-backport
    Found existing installation: eval_type_backport 0.3.1
    Uninstalling eval_type_backport-0.3.1:
      Successfully uninstalled eval_type_backport-0.3.1
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
  Attempting uninstall: mistralai
    Found existing installation: mistralai 2.0.0
    Uninstalling mistralai-2.0.0:
      Successfully uninstalled mistralai-2.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.26.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
firebase-admin 6.9.0 requires

In [ ]:
"""
arXiv PDF Parser — Mistral OCR
===============================
Converts a folder of arXiv PDF files into structured Markdown text,
saving each result as a JSON file. Uses Mistral's vision model (pixtral)
as the OCR/parsing engine.

Designed to run directly in a Google Colab notebook cell.

Step 1 — Install dependencies (in a separate cell, run once):
    !pip install mistralai joblib

Step 2 — Set your Mistral API key in the CONFIG section below.
         Get a free key at: https://console.mistral.ai

Step 3 — Run this script in a Colab cell:
    exec(open('parse_arxiv_pdfs.py').read())
"""

import os
import json
import base64
from joblib import Parallel, delayed, parallel_backend
from mistralai import Mistral


# =============================================================================
# CONFIG — edit these before running
# =============================================================================

MISTRAL_API_KEY = ""
INPUT_DIR       = "/content/drive/MyDrive/arxiv_data/pdf/Computation_and_Language"
OUTPUT_DIR      = "/content/drive/MyDrive/arxiv_data/parsed"
N_PARALLEL_JOBS = 4   # Number of PDFs to process simultaneously.
                      # Keep at 4 for Colab — Mistral free tier has rate limits.

# =============================================================================
# 1. HELPER: Save result as JSON  (replaces the original library helper)
# =============================================================================

def write_json(data: dict, filepath: str) -> None:
    """Save a dictionary as a formatted JSON file."""
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


# =============================================================================
# 2. CORE: Convert one PDF to Markdown via Mistral OCR
#          (replaces the original library OCR class)
# =============================================================================

def convert_pdf_to_markdown(pdf_path: str, client: Mistral) -> dict:
    """
    Read a PDF file and convert its contents to Markdown using Mistral's
    OCR-capable vision model (pixtral-12b).

    The PDF is base64-encoded and sent directly to the API — no upload needed.

    Args:
        pdf_path: Local path to the PDF file.
        client:   An authenticated Mistral API client instance.

    Returns:
        A dictionary with keys:
            'source_file' : original PDF filename
            'markdown'    : extracted text in Markdown format
            'page_count'  : number of pages detected (if available)
    """
    # Read and encode the PDF as base64
    with open(pdf_path, 'rb') as f:
        pdf_bytes = f.read()
    pdf_b64 = base64.standard_b64encode(pdf_bytes).decode('utf-8')

    # Send to Mistral's OCR endpoint
    ocr_response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{pdf_b64}",
        }
    )

    # Combine all pages into a single Markdown string
    pages = ocr_response.pages if hasattr(ocr_response, 'pages') else []
    full_markdown = "\n\n---\n\n".join(
        page.markdown for page in pages if hasattr(page, 'markdown')
    )

    return {
        'source_file': os.path.basename(pdf_path),
        'markdown':    full_markdown,
        'page_count':  len(pages),
    }


# =============================================================================
# 3. PROCESS ONE PDF AND SAVE RESULT
# =============================================================================

def process_and_save_pdf_md(pdf_path: str, output_dir: str) -> None:
    """
    Convert a single PDF to Markdown and save the result as a JSON file.
    Skips the file if the output JSON already exists (safe to re-run).

    Args:
        pdf_path:   Full path to the source PDF file.
        output_dir: Folder where the output JSON will be saved.
    """
    output_path = os.path.join(
        output_dir,
        os.path.basename(pdf_path).replace(".pdf", ".json")
    )

    # Skip if already processed — allows safe re-runs after interruption
    if os.path.exists(output_path):
        print(f"  [SKIP] Already processed: {os.path.basename(pdf_path)}")
        return

    try:
        # Each thread creates its own client instance to avoid shared-state issues
        client = Mistral(api_key=MISTRAL_API_KEY)

        print(f"  [OCR]  Processing: {os.path.basename(pdf_path)}")
        result = convert_pdf_to_markdown(pdf_path, client)
        write_json(result, output_path)
        print(f"  [DONE] Saved: {os.path.basename(output_path)}")

    except Exception as e:
        print(f"  [ERROR] Failed on {os.path.basename(pdf_path)}: {e}")


# =============================================================================
# 4. PARALLEL FOLDER PROCESSOR
# =============================================================================

def iterate_pdfs_parallel(input_directory: str,
                           action,
                           n_jobs: int = 4,
                           **kwargs) -> None:
    """
    Walk through a directory recursively, find all PDF files, and run
    the given action function on each one in parallel.

    Args:
        input_directory: Root folder to search for PDFs.
        action:          Function to call on each PDF path.
        n_jobs:          Number of parallel threads to use.
        **kwargs:        Extra keyword arguments passed to action().
    """
    # Collect all PDF paths first so we can report a total count
    all_pdfs = []
    for root, _, files in os.walk(input_directory):
        for file in files:
            if file.lower().endswith(".pdf"):
                all_pdfs.append(os.path.join(root, file))

    total = len(all_pdfs)
    if total == 0:
        print(f"No PDF files found in: {input_directory}")
        return

    print(f"\nFound {total} PDF files. Starting parallel processing "
          f"with {n_jobs} threads...\n")

    # Process all PDFs in parallel using threads
    # 'threading' backend is used (not multiprocessing) because the bottleneck
    # is network I/O to the Mistral API, not CPU — threads handle this well.
    with parallel_backend('threading', n_jobs=n_jobs):
        Parallel(verbose=5)(
            delayed(action)(pdf_path, **kwargs)
            for pdf_path in all_pdfs
        )


# =============================================================================
# 5. RUN
# =============================================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Count how many are already done (resume support)
already_done = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.json')])
if already_done > 0:
    print(f"Resuming: {already_done} files already parsed, skipping those.\n")

iterate_pdfs_parallel(
    input_directory = INPUT_DIR,
    action          = process_and_save_pdf_md,
    n_jobs          = N_PARALLEL_JOBS,
    output_dir      = OUTPUT_DIR,
)

print("\n=== Parsing complete! ===")
print(f"Output saved to: {OUTPUT_DIR}")

Resuming: 500 files already parsed, skipping those.


Found 500 PDF files. Starting parallel processing with 4 threads...

  [SKIP] Already processed: 2510.07896v3.pdf
  [SKIP] Already processed: 2505.06046v4.pdf
  [SKIP] Already processed: 2509.21344v2.pdf
  [SKIP] Already processed: 2510.02286v2.pdf
  [SKIP] Already processed: 2502.05087v3.pdf
  [SKIP] Already processed: 2512.10054v2.pdf
  [SKIP] Already processed: 2511.08113v3.pdf
  [SKIP] Already processed: 2502.01406v4.pdf
  [SKIP] Already processed: 2510.11892v2.pdf
  [SKIP] Already processed: 2502.13606v2.pdf
  [SKIP] Already processed: 2509.08612v3.pdf
  [SKIP] Already processed: 2511.07405v4.pdf
  [SKIP] Already processed: 2502.17262v4.pdf
  [SKIP] Already processed: 2505.15701v2.pdf
  [SKIP] Already processed: 2502.19747v3.pdf
  [SKIP] Already processed: 2505.07365v2.pdf
  [SKIP] Already processed: 2510.15614v2.pdf
  [SKIP] Already processed: 2505.14996v4.pdf
  [SKIP] Already processed: 2505.13109v5.pdf
  [SKIP] Already proce

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    0.0s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 280 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:    0.2s
[Parallel(n_jobs=4)]: Done 500 out of 500 | elapsed:    0.2s finished


# **3. Content Segmentation**

In [ ]:
"""
arXiv Data Organiser
====================
Converts previously downloaded and parsed arXiv data into the target
corpus format used by the reference RAG benchmark project.

Input  (what you already have):
    arxiv_data/metadata/{paper_id}.json   — paper metadata from arXiv API
    arxiv_data/parsed/{paper_id}.json     — Markdown text from Mistral OCR

Output (what this script produces):
    corpus_output/
        pdf_urls.json          — { paper_id: arxiv_pdf_url, ... }
        corpus/
            {paper_id}.json    — structured paper with sections, tables, images

Designed to run directly in a Google Colab notebook cell.

Step 1 — No extra installs needed (uses only Python standard library).
Step 2 — Edit the CONFIG section below, then run:
    exec(open('organise_corpus.py').read())
"""

import os
import json
import re
import base64
from typing import Optional

# =============================================================================
# CONFIG — edit these paths before running
# =============================================================================

METADATA_DIR  = "/content/drive/MyDrive/arxiv_data/metadata"   # your *.json metadata files
PARSED_DIR    = "/content/drive/MyDrive/arxiv_data/parsed"     # your *.json parsed Markdown files
OUTPUT_DIR    = "/content/drive/MyDrive/arxiv_data/corpus_output"  # where results go


# =============================================================================
# 1. MARKDOWN PARSER
#    Splits one big Markdown string into structured sections,
#    and extracts tables and image placeholders separately.
# =============================================================================

def parse_markdown_into_sections(markdown: str) -> list:
    """
    Split a full-paper Markdown string into a list of section dicts.

    Each section dict has the structure:
        {
            "text":   str,            # section text with ## headings stripped,
                                      # tables replaced by {{TABLE:id}} placeholders,
                                      # images replaced by {{IMAGE:id}} placeholders
            "tables": {id: str, ...}, # raw Markdown table strings keyed by ID
            "images": {id: str, ...}, # base64 image strings keyed by ID (if any)
        }

    Strategy:
      - Split on Markdown heading lines (##, ###, etc.) or the --- page dividers
        inserted by the OCR step. Each chunk becomes one section.
      - Within each chunk, detect Markdown tables (lines starting with |) and
        extract them, replacing each with a {{TABLE:N}} placeholder in the text.
      - Image data is rarely embedded in raw Markdown from PDFs, but the field
        is included in the output for forward compatibility.

    Args:
        markdown: Full Markdown string for one paper.

    Returns:
        List of section dicts. Never returns an empty list — if parsing finds
        nothing, the entire text is returned as a single section.
    """
    if not markdown or not markdown.strip():
        return [{"section_id": "section_0", "text": "", "tables": {}, "images": {}}]

    # ── Step 1: Split into raw chunks on headings or page-break dividers ──────
    # Matches lines that are:  ## Heading, ### Heading, or --- (page dividers)
    split_pattern = re.compile(
        r'(?=^#{1,6}\s.+$)|(?=^---+$)',
        re.MULTILINE
    )
    raw_chunks = split_pattern.split(markdown)

    # Remove empty chunks and strip whitespace
    raw_chunks = [chunk.strip() for chunk in raw_chunks if chunk.strip()]

    if not raw_chunks:
        return [{"text": markdown.strip(), "tables": {}, "images": {}}]

    # ── Step 2: Process each chunk — extract tables and image refs ────────────
    sections = []
    global_table_counter = 0  # Unique table IDs across the whole paper
    global_image_counter = 0  # Unique image IDs across the whole paper

    for chunk in raw_chunks:
        tables = {}
        images = {}
        processed_lines = []

        # Collect consecutive table lines into one table block
        table_buffer = []

        def flush_table_buffer():
            """Save buffered table lines as a table entry and add placeholder."""
            nonlocal global_table_counter
            if not table_buffer:
                return
            table_id = f"table_{global_table_counter}"
            tables[table_id] = "\n".join(table_buffer)
            processed_lines.append(f"{{{{TABLE:{table_id}}}}}")
            table_buffer.clear()
            global_table_counter += 1

        for line in chunk.splitlines():
            stripped = line.strip()

            # ── Detect Markdown table rows (start with |) ─────────────────
            if stripped.startswith("|"):
                table_buffer.append(line)
                continue

            # ── Non-table line: flush any pending table buffer first ───────
            if table_buffer:
                flush_table_buffer()

            # ── Detect Markdown image syntax: ![alt](url_or_data) ─────────
            img_match = re.search(r'!\[([^\]]*)\]\(([^)]+)\)', stripped)
            if img_match:
                img_src = img_match.group(2)
                image_id = f"image_{global_image_counter}"

                # If it's a base64 data URI, extract the raw base64 string
                if img_src.startswith("data:"):
                    b64_match = re.search(r'base64,(.+)', img_src)
                    images[image_id] = b64_match.group(1) if b64_match else img_src
                else:
                    # For URL references, store the URL as a placeholder
                    images[image_id] = img_src

                # Replace the full image tag with a placeholder in the text
                line = re.sub(
                    r'!\[([^\]]*)\]\([^)]+\)',
                    f"{{{{IMAGE:{image_id}}}}}",
                    line
                )
                global_image_counter += 1

            processed_lines.append(line)

        # Flush any table that runs to the end of the chunk
        if table_buffer:
            flush_table_buffer()

        section_text = "\n".join(processed_lines).strip()

        # Only add non-empty sections
        if section_text or tables or images:
            sections.append({
                "section_id": f"section_{len(sections)}",
                "text":       section_text,
                "tables":     tables,
                "images":     images,
            })

    return sections if sections else [{"section_id": "section_0", "text": markdown.strip(), "tables": {}, "images": {}}]


# =============================================================================
# 2. PDF URL BUILDER
# =============================================================================

def build_pdf_url(paper_id: str) -> str:
    """
    Reconstruct the arXiv PDF URL from a paper ID.

    arXiv PDF URLs follow a consistent pattern:
        https://arxiv.org/pdf/{paper_id}

    This works for both old-style IDs (e.g. "hep-th/9901001") and
    new-style IDs (e.g. "2401.12345v2").

    Args:
        paper_id: The arXiv paper ID string.

    Returns:
        Full HTTPS URL to the paper's PDF.
    """
    return f"https://arxiv.org/pdf/{paper_id}"


# =============================================================================
# 3. SINGLE PAPER CONVERTER
# =============================================================================

def build_corpus_entry(metadata: dict, parsed: dict) -> dict:
    """
    Merge one paper's metadata and parsed Markdown into the target corpus format.

    Target format:
        {
            "title":      str,
            "sections":   [ {text, tables, images}, ... ],
            "id":         str,
            "authors":    [str, ...],
            "categories": [str, ...],
            "abstract":   str,
            "updated":    str,
            "published":  str,
        }

    Args:
        metadata: Dict loaded from metadata/{paper_id}.json
        parsed:   Dict loaded from parsed/{paper_id}.json

    Returns:
        Corpus entry dict in the target format.
    """
    markdown_text = parsed.get("markdown", "")
    sections = parse_markdown_into_sections(markdown_text)

    for i, section in enumerate(sections):
        section["section_id"] = i

    for i, section in enumerate(sections):
        section["section_id"] = i

    return {
        "title":      metadata.get("title", ""),
        "sections":   sections,
        "id":         metadata.get("id", ""),
        "authors":    metadata.get("authors", []),
        "categories": metadata.get("categories", []),
        "abstract":   metadata.get("abstract", ""),
        "updated":    metadata.get("updated", ""),
        "published":  metadata.get("published", ""),
    }


# =============================================================================
# 4. MAIN ORGANISER
# =============================================================================

def organise_corpus(metadata_dir: str,
                    parsed_dir: str,
                    output_dir: str) -> None:
    """
    Process all papers and write pdf_urls.json + corpus/ directory.

    Matching logic:
      - For each metadata file, look for a corresponding parsed file
        with the same paper ID (same filename stem).
      - Papers with no parsed file are listed in a warning but still
        get a pdf_urls.json entry (so you know what wasn't processed).
      - Corpus entries are only written when both files exist.

    Args:
        metadata_dir: Folder containing {paper_id}.json metadata files.
        parsed_dir:   Folder containing {paper_id}.json parsed Markdown files.
        output_dir:   Root folder for all output (created if missing).
    """
    corpus_dir = os.path.join(output_dir, "corpus")
    os.makedirs(corpus_dir, exist_ok=True)

    # ── Collect all available paper IDs from metadata ─────────────────────────
    metadata_files = {
        fname.replace(".json", ""): os.path.join(metadata_dir, fname)
        for fname in os.listdir(metadata_dir)
        if fname.endswith(".json")
    }

    # ── Collect all available parsed files ───────────────────────────────────
    parsed_files = {
        fname.replace(".json", ""): os.path.join(parsed_dir, fname)
        for fname in os.listdir(parsed_dir)
        if fname.endswith(".json")
    }

    total    = len(metadata_files)
    matched  = 0
    skipped  = 0
    errors   = 0
    pdf_urls = {}

    print(f"Found {total} metadata files.")
    print(f"Found {len(parsed_files)} parsed files.")
    print(f"\nStarting conversion...\n")

    for paper_id, meta_path in metadata_files.items():

        # ── Load metadata ─────────────────────────────────────────────────────
        try:
            with open(meta_path, 'r', encoding='utf-8') as f:
                metadata = json.load(f)
        except Exception as e:
            print(f"  [ERROR] Could not read metadata for {paper_id}: {e}")
            errors += 1
            continue

        # ── Always record the PDF URL (even if parsed file is missing) ────────
        pdf_urls[paper_id] = build_pdf_url(paper_id)

        # ── Check for corresponding parsed file ───────────────────────────────
        if paper_id not in parsed_files:
            print(f"  [SKIP]  No parsed file for: {paper_id}")
            skipped += 1
            continue

        # ── Skip if corpus entry already exists (resume support) ──────────────
        corpus_path = os.path.join(corpus_dir, f"{paper_id}.json")
        if os.path.exists(corpus_path):
            print(f"  [SKIP]  Corpus entry already exists: {paper_id}")
            matched += 1
            continue

        # ── Load parsed Markdown ──────────────────────────────────────────────
        try:
            with open(parsed_files[paper_id], 'r', encoding='utf-8') as f:
                parsed = json.load(f)
        except Exception as e:
            print(f"  [ERROR] Could not read parsed file for {paper_id}: {e}")
            errors += 1
            continue

        # ── Build and save corpus entry ───────────────────────────────────────
        try:
            corpus_entry = build_corpus_entry(metadata, parsed)
            with open(corpus_path, 'w', encoding='utf-8') as f:
                json.dump(corpus_entry, f, indent=2, ensure_ascii=False)
            print(f"  [DONE]  {paper_id}  "
                  f"({len(corpus_entry['sections'])} sections)")
            matched += 1
        except Exception as e:
            print(f"  [ERROR] Failed to build corpus entry for {paper_id}: {e}")
            errors += 1

    # ── Save pdf_urls.json ────────────────────────────────────────────────────
    pdf_urls_path = os.path.join(output_dir, "pdf_urls.json")
    with open(pdf_urls_path, 'w', encoding='utf-8') as f:
        json.dump(pdf_urls, f, indent=2, ensure_ascii=False)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  CONVERSION COMPLETE")
    print(f"{'='*55}")
    print(f"  Total metadata files:   {total}")
    print(f"  Corpus entries written: {matched}")
    print(f"  Skipped (no parsed):    {skipped}")
    print(f"  Errors:                 {errors}")
    print(f"\n  pdf_urls.json  →  {pdf_urls_path}")
    print(f"  corpus/        →  {corpus_dir}")
    print(f"{'='*55}\n")


# =============================================================================
# 5. RUN
# =============================================================================

organise_corpus(
    metadata_dir = METADATA_DIR,
    parsed_dir   = PARSED_DIR,
    output_dir   = OUTPUT_DIR,
)

Found 500 metadata files.
Found 500 parsed files.

Starting conversion...

  [SKIP]  Corpus entry already exists: 2510.02286v2
  [SKIP]  Corpus entry already exists: 2509.21344v2
  [SKIP]  Corpus entry already exists: 2510.07896v3
  [SKIP]  Corpus entry already exists: 2505.15701v2
  [SKIP]  Corpus entry already exists: 2505.06046v4
  [SKIP]  Corpus entry already exists: 2510.11892v2
  [SKIP]  Corpus entry already exists: 2502.01406v4
  [SKIP]  Corpus entry already exists: 2502.05087v3
  [SKIP]  Corpus entry already exists: 2512.10054v2
  [SKIP]  Corpus entry already exists: 2511.08113v3
  [SKIP]  Corpus entry already exists: 2509.08612v3
  [SKIP]  Corpus entry already exists: 2511.07405v4
  [SKIP]  Corpus entry already exists: 2502.13606v2
  [SKIP]  Corpus entry already exists: 2502.17262v4
  [SKIP]  Corpus entry already exists: 2502.19747v3
  [SKIP]  Corpus entry already exists: 2505.07365v2
  [SKIP]  Corpus entry already exists: 2505.14996v4
  [SKIP]  Corpus entry already exists: 25

# **4. QA Pairs Generation**

In [ ]:
"""
QA Benchmark Generation Script
================================
Builds a 1000-pair QA evaluation benchmark from 500 arXiv corpus papers.

Pipeline:
  Step 1 — Randomly split 500 papers into 200 positive + 300 hard negative docs
  Step 2 — Sample sections from positive docs, balancing content types
  Step 3 — Generate QA pairs via deepseek-chat (question + answer + type tags)
  Step 4 — Save all outputs in reference-project format

Output files (all written to OUTPUT_DIR):
  queries.json       — { query_id: { query, type, source }, ... }
  answers.json       — { query_id: answer_text, ... }
  qrels.json         — { query_id: { doc_id, section_id }, ... }
  corpus_split.json  — { "positive": [...ids], "hard_negative": [...ids] }
  benchmark.csv      — question, answer, query_type, source_type, paper_id, section_id

query_id format: {paper_id}_s{section_id}

Target QA distribution (1000 pairs total):
  By query type:   ~450 abstractive,  ~550 extractive
  By source type:  800 text-only, 200 text-table

Designed to run in Google Colab. No argparse — edit CONFIG below.

Step 1:
    !pip install openai tqdm

Step 2:
    Edit DEEPSEEK_API_KEY in CONFIG, then run:
    exec(open('generate_benchmark.py').read())
"""

import os
import json
import csv
import re
import time
import random
from tqdm import tqdm
from openai import OpenAI

# =============================================================================
# CONFIG — edit these before running
# =============================================================================

DEEPSEEK_API_KEY   = ""

CORPUS_DIR         = "/content/drive/MyDrive/arxiv_data/corpus_output/corpus"
OUTPUT_DIR         = "/content/drive/MyDrive/arxiv_data/corpus_output"

RANDOM_SEED        = 42      # for reproducibility of the 200/300 split
N_POSITIVE         = 200     # number of positive documents
N_QA_PAIRS         = 1000    # total QA pairs to generate
MIN_SECTION_WORDS  = 50      # skip sections shorter than this
MAX_SECTION_WORDS  = 400     # truncate sections longer than this for API calls

SLEEP_BETWEEN_CALLS = 0.5    # seconds between API calls
MAX_RETRIES         = 5      # retries per section on API error
RETRY_WAIT          = 10     # seconds to wait before each retry

# Target source-type distribution (must sum to N_QA_PAIRS)
TARGET_DISTRIBUTION = {
    "text":       800,
    "text-table": 200,
}

# =============================================================================
# PROMPTS
# =============================================================================

SYSTEM_PROMPT = """You are an expert at creating question-answer pairs for evaluating academic RAG systems.
Given a section of an academic paper (and optionally its tables/images description), generate exactly ONE question-answer pair.

You must also classify the pair along two dimensions:

1. query_type — choose ONE:
   - "abstractive": the answer requires synthesising or summarising information (cannot be copied word-for-word)
   - "extractive":  the answer is a specific fact, number, name, or phrase directly stated in the text

2. source_type — choose ONE based on what content the answer actually relies on:
   - "text":        answer comes from the text only
   - "text-table":  answer requires reading a table in the section

Rules for the question:
   - Must be answerable using ONLY the provided section content
   - Must be specific enough that only this section could answer it
   - Should sound like something a researcher would genuinely ask
   - Length: 10 to 25 words

Rules for the answer:
   - Must be accurate and grounded in the section content
   - Abstractive answers: 2–4 sentences in your own words
   - Extractive answers: 1 sentence, as concise as possible

Respond in this exact JSON format (no extra text, no markdown fences):
{
  "question": "...",
  "answer": "...",
  "query_type": "abstractive" or "extractive",
  "source_type": "text" or "text-table"
}"""

USER_PROMPT_TEMPLATE = """Section text:
{section_text}

{table_info}Generate one question-answer pair for this section:"""


# =============================================================================
# HELPERS
# =============================================================================

def count_words(text: str) -> int:
    return len(text.split())


def truncate_text(text: str, max_words: int = MAX_SECTION_WORDS) -> str:
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + " ..."


def clean_section_text(text: str) -> str:
    """Remove TABLE/IMAGE placeholders and normalise whitespace."""
    text = re.sub(r'\{\{TABLE:[^}]+\}\}', '', text)
    text = re.sub(r'\{\{IMAGE:[^}]+\}\}', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def get_section_content_type(section: dict) -> str:
    """
    Determine what multimodal content a section contains.
    Since only text and text-table sources are used, sections with images
    (but no table) are classified as 'text', and sections with both a
    table and images are classified as 'text-table'.
    Returns one of: 'text', 'text-table'
    """
    has_table = bool(section.get("tables"))
    return "text-table" if has_table else "text"


def build_table_info(section: dict) -> str:
    """Format table content for the prompt (first table only, truncated)."""
    tables = section.get("tables", {})
    if not tables:
        return ""
    first_table = next(iter(tables.values()))
    # Truncate very long tables to first 20 lines
    lines = first_table.strip().splitlines()[:20]
    truncated = "\n".join(lines)
    if len(first_table.strip().splitlines()) > 20:
        truncated += "\n... (table truncated)"
    return f"Table in this section:\n{truncated}\n\n"




# =============================================================================
# STEP 1: SPLIT CORPUS INTO POSITIVE / HARD NEGATIVE
# =============================================================================

def split_corpus(corpus_dir: str, n_positive: int, seed: int) -> tuple:
    """
    Randomly split all papers in corpus_dir into positive and hard-negative sets.

    Args:
        corpus_dir: Folder containing {paper_id}.json corpus files.
        n_positive: Number of papers to designate as positive documents.
        seed:       Random seed for reproducibility.

    Returns:
        (positive_ids, hard_negative_ids) — two lists of paper ID strings.
    """
    all_ids = sorted([
        f.replace(".json", "")
        for f in os.listdir(corpus_dir)
        if f.endswith(".json")
    ])

    if len(all_ids) < n_positive:
        raise ValueError(
            f"Corpus has only {len(all_ids)} papers, "
            f"but {n_positive} positives were requested."
        )

    random.seed(seed)
    random.shuffle(all_ids)

    positive_ids      = all_ids[:n_positive]
    hard_negative_ids = all_ids[n_positive:]

    print(f"Corpus split:")
    print(f"  Total papers:    {len(all_ids)}")
    print(f"  Positive docs:   {len(positive_ids)}")
    print(f"  Hard negatives:  {len(hard_negative_ids)}")

    return positive_ids, hard_negative_ids


# =============================================================================
# STEP 2: SAMPLE SECTIONS FROM POSITIVE DOCUMENTS
# =============================================================================

def sample_sections(corpus_dir: str,
                    positive_ids: list,
                    target_distribution: dict,
                    min_words: int) -> list:
    """
    Select sections from positive documents to match the target type distribution.

    Strategy:
      - For each content type (text, text-table), collect all eligible
        sections from positive documents.
      - Randomly sample from each pool to hit the target count.
      - Falls back to text-only sections if text-table is under-represented.

    Args:
        corpus_dir:          Folder containing corpus JSON files.
        positive_ids:        List of paper IDs designated as positive documents.
        target_distribution: Dict mapping content type → desired count.
        min_words:           Minimum word count for a section to be eligible.

    Returns:
        List of dicts, each containing:
            { paper_id, section, content_type }
    """
    # Collect all eligible sections grouped by content type
    pools = {"text": [], "text-table": []}

    print(f"\nScanning {len(positive_ids)} positive documents for sections...")

    for paper_id in tqdm(positive_ids, desc="Scanning"):
        fpath = os.path.join(corpus_dir, f"{paper_id}.json")
        try:
            with open(fpath, 'r', encoding='utf-8') as f:
                paper = json.load(f)
        except Exception as e:
            print(f"  [WARN] Could not read {paper_id}: {e}")
            continue

        for section in paper.get("sections", []):
            clean_text = clean_section_text(section.get("text", ""))
            if count_words(clean_text) < min_words:
                continue
            content_type = get_section_content_type(section)
            pools[content_type].append({
                "paper_id":     paper_id,
                "section":      section,
                "content_type": content_type,
            })

    # Report pool sizes
    print(f"\n  Available sections by type:")
    for ctype, pool in pools.items():
        target = target_distribution.get(ctype, 0)
        status = "OK" if len(pool) >= target else f"SHORTAGE — need {target}, have {len(pool)}"
        print(f"    {ctype:<22} {len(pool):>5} available  (target: {target})  {status}")

    # Sample from each pool
    random.seed(RANDOM_SEED + 1)  # different seed from the corpus split
    selected = []

    for ctype, target_count in target_distribution.items():
        pool = pools[ctype]
        if len(pool) >= target_count:
            sampled = random.sample(pool, target_count)
        else:
            # Not enough sections of this type — use all available, log warning
            print(f"\n  [WARN] Only {len(pool)} {ctype} sections available (wanted {target_count}).")
            print(f"         Using all {len(pool)} and padding remainder with text-only sections.")
            sampled = pool
            # Pad shortage from text-only pool (largest pool)
            shortage = target_count - len(pool)
            padding = random.sample(pools["text"], min(shortage, len(pools["text"])))
            sampled += padding

        selected.extend(sampled)

    print(f"\nTotal sections selected: {len(selected)}")
    return selected


# =============================================================================
# STEP 3: GENERATE QA PAIRS
# =============================================================================

def generate_qa_pair(client: OpenAI, section: dict, content_type: str) -> dict:
    """
    Call deepseek-chat to generate a QA pair for one section.

    The model returns a JSON object with:
        question, answer, query_type, source_type

    We override source_type with the actual detected content_type to ensure
    it matches what the section really contains (the model might misclassify).

    Args:
        client:       Initialised OpenAI client (pointing to DeepSeek base_url).
        section:      Section dict from corpus.
        content_type: Detected content type for this section.

    Returns:
        Dict with question, answer, query_type, source_type,
        or empty dict if all retries fail.
    """
    clean_text     = clean_section_text(section.get("text", ""))
    truncated_text = truncate_text(clean_text)
    table_info     = build_table_info(section) if "table" in content_type else ""

    user_prompt = USER_PROMPT_TEMPLATE.format(
        section_text=truncated_text,
        table_info=table_info,
    )

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=0.7,
                max_tokens=300,
                response_format={"type": "json_object"},  # enforce JSON output
            )

            raw    = response.choices[0].message.content.strip()
            parsed = json.loads(raw)

            # Validate required fields
            for field in ("question", "answer", "query_type", "source_type"):
                if field not in parsed:
                    raise ValueError(f"Missing field: {field}")

            # Normalise query_type
            if parsed["query_type"] not in ("abstractive", "extractive"):
                parsed["query_type"] = "extractive"

            # Override source_type with ground truth from actual section content
            parsed["source_type"] = content_type

            return parsed

        except Exception as e:
            print(f"      [Attempt {attempt}/{MAX_RETRIES}] Error: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT)
            else:
                print(f"      All retries exhausted.")
                return {}


# =============================================================================
# STEP 4: LOAD CHECKPOINT (resume support)
# =============================================================================

def load_checkpoint(output_dir: str) -> dict:
    """
    Load previously generated results from disk.
    Returns a dict keyed by query_id.
    """
    checkpoint_path = os.path.join(output_dir, "benchmark_checkpoint.json")
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        print(f"  Resumed: loaded {len(data)} existing QA pairs from checkpoint.")
        return data
    return {}


def save_checkpoint(output_dir: str, results: dict):
    """Save current results to checkpoint file."""
    checkpoint_path = os.path.join(output_dir, "benchmark_checkpoint.json")
    with open(checkpoint_path, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)


# =============================================================================
# STEP 5: SAVE ALL FINAL OUTPUTS
# =============================================================================

def save_all_outputs(output_dir: str,
                     results: dict,
                     positive_ids: list,
                     hard_negative_ids: list):
    """
    Write all output files from the accumulated results dict.

    Args:
        output_dir:         Where to write output files.
        results:            Dict keyed by query_id, each value containing
                            paper_id, section_id, question, answer,
                            query_type, source_type.
        positive_ids:       List of positive document paper IDs.
        hard_negative_ids:  List of hard negative document paper IDs.
    """
    # ── queries.json ──────────────────────────────────────────────────────────
    queries = {
        qid: {
            "query":  v["question"],
            "type":   v["query_type"],
            "source": v["source_type"],
        }
        for qid, v in results.items()
    }
    with open(os.path.join(output_dir, "queries.json"), 'w', encoding='utf-8') as f:
        json.dump(queries, f, indent=2, ensure_ascii=False)

    # ── answers.json ──────────────────────────────────────────────────────────
    answers = {qid: v["answer"] for qid, v in results.items()}
    with open(os.path.join(output_dir, "answers.json"), 'w', encoding='utf-8') as f:
        json.dump(answers, f, indent=2, ensure_ascii=False)

    # ── qrels.json ────────────────────────────────────────────────────────────
    qrels = {
        qid: {
            "doc_id":     v["paper_id"],
            "section_id": v["section_id"],
        }
        for qid, v in results.items()
    }
    with open(os.path.join(output_dir, "qrels.json"), 'w', encoding='utf-8') as f:
        json.dump(qrels, f, indent=2, ensure_ascii=False)

    # ── corpus_split.json ─────────────────────────────────────────────────────
    corpus_split = {
        "positive":      positive_ids,
        "hard_negative": hard_negative_ids,
    }
    with open(os.path.join(output_dir, "corpus_split.json"), 'w', encoding='utf-8') as f:
        json.dump(corpus_split, f, indent=2, ensure_ascii=False)

    # ── benchmark.csv ─────────────────────────────────────────────────────────
    csv_path = os.path.join(output_dir, "benchmark.csv")
    fieldnames = ["query_id", "question", "answer", "query_type",
                  "source_type", "paper_id", "section_id"]
    with open(csv_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for qid, v in results.items():
            writer.writerow({
                "query_id":   qid,
                "question":   v["question"],
                "answer":     v["answer"],
                "query_type": v["query_type"],
                "source_type": v["source_type"],
                "paper_id":   v["paper_id"],
                "section_id": v["section_id"],
            })

    print(f"\n  queries.json      →  {os.path.join(output_dir, 'queries.json')}")
    print(f"  answers.json      →  {os.path.join(output_dir, 'answers.json')}")
    print(f"  qrels.json        →  {os.path.join(output_dir, 'qrels.json')}")
    print(f"  corpus_split.json →  {os.path.join(output_dir, 'corpus_split.json')}")
    print(f"  benchmark.csv     →  {csv_path}")


# =============================================================================
# MAIN
# =============================================================================

def generate_benchmark(corpus_dir: str, output_dir: str) -> None:
    """
    Full pipeline: split corpus → sample sections → generate QA → save outputs.
    """
    os.makedirs(output_dir, exist_ok=True)
    client = OpenAI(
        api_key  = DEEPSEEK_API_KEY,
        base_url = "https://api.deepseek.com",
    )

    # ── Step 1: Split corpus ──────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STEP 1: Splitting corpus")
    print(f"{'='*55}")

    split_path = os.path.join(output_dir, "corpus_split.json")
    if os.path.exists(split_path):
        # Reuse existing split for reproducibility if script is rerun
        with open(split_path, 'r', encoding='utf-8') as f:
            split = json.load(f)
        positive_ids      = split["positive"]
        hard_negative_ids = split["hard_negative"]
        print(f"  Loaded existing split: "
              f"{len(positive_ids)} positive, {len(hard_negative_ids)} hard negative.")
    else:
        positive_ids, hard_negative_ids = split_corpus(
            corpus_dir, N_POSITIVE, RANDOM_SEED
        )

    # ── Step 2: Sample sections ───────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STEP 2: Sampling sections")
    print(f"{'='*55}")

    selected_sections = sample_sections(
        corpus_dir, positive_ids, TARGET_DISTRIBUTION, MIN_SECTION_WORDS
    )

    # ── Step 3: Generate QA pairs ─────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STEP 3: Generating QA pairs")
    print(f"{'='*55}\n")

    results       = load_checkpoint(output_dir)

    # ── Trim checkpoint to only validated pairs ───────────────────────────────
    # If benchmark.csv already exists (from a previous run), treat it as the
    # ground truth of validated pairs. Any query_ids in the checkpoint but NOT
    # in benchmark.csv were either flagged for deletion or came from a bad run
    # and should be discarded, forcing regeneration.
    benchmark_csv_path = os.path.join(output_dir, "benchmark.csv")
    if os.path.exists(benchmark_csv_path):
        with open(benchmark_csv_path, 'r', encoding='utf-8') as f:
            validated_ids = {
                row.split(',')[0]
                for row in f.read().splitlines()[1:]  # skip header
                if row.strip()
            }
        before = len(results)
        results = {k: v for k, v in results.items() if k in validated_ids}
        trimmed = before - len(results)
        if trimmed > 0:
            print(f"  Trimmed {trimmed} unvalidated entries from checkpoint.")
            print(f"  Retained {len(results)} validated pairs from benchmark.csv.")
        save_checkpoint(output_dir, results)  # persist the trimmed checkpoint

    already_done  = set(results.keys())
    generated     = 0
    failed        = 0

    for item in tqdm(selected_sections, desc="Generating QA pairs"):
        paper_id     = item["paper_id"]
        section      = item["section"]
        content_type = item["content_type"]
        section_id   = section.get("section_id", 0)
        query_id     = f"{paper_id}_s{section_id}"

        # Skip if already generated (resume support)
        if query_id in already_done:
            continue

        qa = generate_qa_pair(client, section, content_type)

        if not qa:
            failed += 1
            continue

        results[query_id] = {
            "paper_id":   paper_id,
            "section_id": section_id,
            "question":   qa["question"],
            "answer":     qa["answer"],
            "query_type": qa["query_type"],
            "source_type": qa["source_type"],
        }
        generated += 1
        time.sleep(SLEEP_BETWEEN_CALLS)

        # Checkpoint every 10 pairs
        if generated % 10 == 0:
            save_checkpoint(output_dir, results)

    # Final checkpoint
    save_checkpoint(output_dir, results)

    # ── Step 4: Save all outputs ──────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STEP 4: Saving outputs")
    print(f"{'='*55}")

    save_all_outputs(output_dir, results, positive_ids, hard_negative_ids)

    # ── Summary ───────────────────────────────────────────────────────────────
    query_type_counts  = {}
    source_type_counts = {}
    for v in results.values():
        qt = v.get("query_type", "unknown")
        st = v.get("source_type", "unknown")
        query_type_counts[qt]  = query_type_counts.get(qt, 0) + 1
        source_type_counts[st] = source_type_counts.get(st, 0) + 1

    print(f"\n{'='*55}")
    print(f"  BENCHMARK GENERATION COMPLETE")
    print(f"{'='*55}")
    print(f"  Total QA pairs generated:  {len(results)}")
    print(f"  Failed:                    {failed}")
    print(f"\n  By query type:")
    for k, v in sorted(query_type_counts.items()):
        print(f"    {k:<16} {v}")
    print(f"\n  By source type:")
    for k, v in sorted(source_type_counts.items()):
        print(f"    {k:<22} {v}")
    print(f"{'='*55}\n")


# =============================================================================
# RUN
# =============================================================================

generate_benchmark(
    corpus_dir = CORPUS_DIR,
    output_dir = OUTPUT_DIR,
)


  STEP 1: Splitting corpus
  Loaded existing split: 200 positive, 300 hard negative.

  STEP 2: Sampling sections

Scanning 200 positive documents for sections...


Scanning: 100%|██████████| 200/200 [00:04<00:00, 49.78it/s]



  Available sections by type:
    text                    9442 available  (target: 800)  OK
    text-table              1272 available  (target: 200)  OK

Total sections selected: 1000

  STEP 3: Generating QA pairs

  Resumed: loaded 1000 existing QA pairs from checkpoint.


Generating QA pairs: 100%|██████████| 1000/1000 [00:00<00:00, 444264.80it/s]



  STEP 4: Saving outputs

  queries.json      →  /content/drive/MyDrive/arxiv_data/corpus_output/queries.json
  answers.json      →  /content/drive/MyDrive/arxiv_data/corpus_output/answers.json
  qrels.json        →  /content/drive/MyDrive/arxiv_data/corpus_output/qrels.json
  corpus_split.json →  /content/drive/MyDrive/arxiv_data/corpus_output/corpus_split.json
  benchmark.csv     →  /content/drive/MyDrive/arxiv_data/corpus_output/benchmark.csv

  BENCHMARK GENERATION COMPLETE
  Total QA pairs generated:  1000
  Failed:                    0

  By query type:
    abstractive      357
    extractive       643

  By source type:
    text                   800
    text-table             200



# **5. Quality Filtering**

In [ ]:
!pip install openai tqdm rapidfuzz

In [ ]:
"""
QA Benchmark Quality Filter
============================
Applies two-stage quality filtering to the 1000-pair QA benchmark generated
by generate_benchmark.py.

Stage 1 — Rule-based filter (free, instant):
  Flags and removes pairs that fail structural/content sanity checks.

Stage 2 — LLM-based filter (DeepSeek API):
  Scores each remaining pair on three dimensions (1–3 scale):
    - Clarity:       Is the question well-formed and unambiguous?
    - Accuracy:      Is the answer grounded in the section content?
    - Retrievability: Could this question retrieve its source section?
  Threshold: score of 2 used for all dimensions (balanced).
  A pair is kept only if ALL three scores >= 2.

Output files (saved alongside originals, originals are NOT modified):
  queries_filtered.json
  answers_filtered.json
  qrels_filtered.json
  benchmark_filtered.csv
  filter_report.json    — full audit log of every decision made

Designed to run in Google Colab. No argparse — edit CONFIG below.

Step 1:
    !pip install openai tqdm rapidfuzz

Step 2:
    Edit DEEPSEEK_API_KEY in CONFIG, then run:
    exec(open('filter_benchmark.py').read())
"""

import os
import json
import csv
import re
import time
from tqdm import tqdm
from openai import OpenAI
from rapidfuzz import fuzz

# =============================================================================
# CONFIG — edit these before running
# =============================================================================

DEEPSEEK_API_KEY = ""

INPUT_DIR  = "/content/drive/MyDrive/arxiv_data/corpus_output"
OUTPUT_DIR = "/content/drive/MyDrive/arxiv_data/corpus_output"

# Stage 1 thresholds
MIN_QUESTION_WORDS  = 8     # questions shorter than this are rejected
MAX_QUESTION_WORDS  = 40    # questions longer than this are rejected
MIN_ANSWER_WORDS    = 5     # answers shorter than this are rejected
DEDUP_THRESHOLD     = 90    # fuzzy similarity % above which a pair is a duplicate

# Stage 2 thresholds (all three scores must meet this minimum to pass)
LLM_MIN_SCORE = 2           # 1=strict, 2=balanced, 3=lenient

# API settings
SLEEP_BETWEEN_CALLS = 0.5
MAX_RETRIES         = 5
RETRY_WAIT          = 10


# =============================================================================
# STAGE 1: RULE-BASED FILTER
# =============================================================================

# Phrases that make a question non-retrievable (requires seeing the source)
NON_RETRIEVABLE_PATTERNS = [
    r'\bthis section\b',
    r'\bthe (above|following|given|provided) (text|section|passage|paragraph)\b',
    r'\baccording to (the )?(above|this|given|provided)\b',
    r'\bin (this|the above) (text|passage|section|paragraph)\b',
    r'\bthe author(s)? (state|mention|describe|note|claim|argue|write)s? (here|above|in this)\b',
]

# Phrases indicating a model refusal leaked into the answer
REFUSAL_PATTERNS = [
    r'\bi (cannot|can\'t|could not|couldn\'t)\b',
    r'\bi don\'t (know|have|see)\b',
    r'\bas an ai\b',
    r'\bi (am|\'m) (unable|not able)\b',
    r'\bno (information|content|text|data) (is |was )?(provided|given|available)\b',
]


def stage1_filter(pairs: dict) -> tuple:
    """
    Apply rule-based checks to all QA pairs.

    Checks (in order):
      1. Question word count within bounds
      2. Answer word count above minimum
      3. Question does not contain non-retrievable phrases
      4. Answer does not contain model refusal phrases
      5. Question is not a near-duplicate of a previously seen question

    Args:
        pairs: Dict keyed by query_id, each value containing
               question, answer, query_type, source_type, paper_id, section_id.

    Returns:
        (passed, rejected) — two dicts with the same structure as pairs.
        Each rejected entry has an extra 'reject_reason' key.
    """
    passed   = {}
    rejected = {}
    seen_questions = []   # for deduplication

    non_retrievable_re = re.compile(
        '|'.join(NON_RETRIEVABLE_PATTERNS), re.IGNORECASE
    )
    refusal_re = re.compile(
        '|'.join(REFUSAL_PATTERNS), re.IGNORECASE
    )

    for qid, pair in pairs.items():
        question = pair.get("question", "").strip()
        answer   = pair.get("answer",   "").strip()
        reason   = None

        # ── Check 1: question length ──────────────────────────────────────────
        q_words = len(question.split())
        if q_words < MIN_QUESTION_WORDS:
            reason = f"question too short ({q_words} words, min {MIN_QUESTION_WORDS})"
        elif q_words > MAX_QUESTION_WORDS:
            reason = f"question too long ({q_words} words, max {MAX_QUESTION_WORDS})"

        # ── Check 2: answer length ────────────────────────────────────────────
        elif len(answer.split()) < MIN_ANSWER_WORDS:
            reason = f"answer too short ({len(answer.split())} words, min {MIN_ANSWER_WORDS})"

        # ── Check 3: non-retrievable question ─────────────────────────────────
        elif non_retrievable_re.search(question):
            match = non_retrievable_re.search(question).group(0)
            reason = f"non-retrievable phrase in question: \"{match}\""

        # ── Check 4: model refusal in answer ──────────────────────────────────
        elif refusal_re.search(answer):
            match = refusal_re.search(answer).group(0)
            reason = f"refusal phrase in answer: \"{match}\""

        # ── Check 5: near-duplicate question ──────────────────────────────────
        else:
            for prev_q in seen_questions:
                similarity = fuzz.ratio(question.lower(), prev_q.lower())
                if similarity >= DEDUP_THRESHOLD:
                    reason = f"near-duplicate question (similarity {similarity:.0f}% >= {DEDUP_THRESHOLD}%)"
                    break

        if reason:
            rejected[qid] = {**pair, "reject_reason": reason, "filter_stage": 1}
        else:
            passed[qid] = pair
            seen_questions.append(question)

    return passed, rejected


# =============================================================================
# STAGE 2: LLM-BASED FILTER
# =============================================================================

SCORE_SYSTEM_PROMPT = """You are a strict quality evaluator for a RAG benchmark dataset.
You will be given a question-answer pair from an academic paper section.
Score the pair on three dimensions, each from 1 to 3:

  clarity       — Is the question specific, well-formed, and unambiguous?
                  1 = vague or confusing
                  2 = acceptable, minor issues
                  3 = clear and specific

  accuracy      — Is the answer accurate and fully grounded in the question?
                  1 = incorrect, hallucinated, or off-topic
                  2 = mostly correct, minor gaps
                  3 = accurate and complete

  retrievability — Could a researcher use this question to find the source section?
                  1 = too generic, many sections could match
                  2 = somewhat specific, likely to retrieve correctly
                  3 = highly specific, uniquely points to source

Respond ONLY in this exact JSON format (no extra text):
{
  "clarity": 1 or 2 or 3,
  "accuracy": 1 or 2 or 3,
  "retrievability": 1 or 2 or 3,
  "reason": "one sentence explaining the lowest score"
}"""

SCORE_USER_TEMPLATE = """Question: {question}

Answer: {answer}

Score this question-answer pair:"""


def score_pair(client: OpenAI, question: str, answer: str) -> dict:
    """
    Ask DeepSeek to score a single QA pair on clarity, accuracy, retrievability.

    Args:
        client:   Initialised OpenAI client pointing to DeepSeek.
        question: The question string.
        answer:   The answer string.

    Returns:
        Dict with clarity, accuracy, retrievability (int 1–3) and reason (str),
        or empty dict if all retries fail.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": SCORE_SYSTEM_PROMPT},
                    {"role": "user",   "content": SCORE_USER_TEMPLATE.format(
                        question=question,
                        answer=answer,
                    )},
                ],
                temperature=0.0,    # deterministic scoring
                max_tokens=100,
                response_format={"type": "json_object"},
            )

            raw    = response.choices[0].message.content.strip()
            parsed = json.loads(raw)

            # Validate and clamp scores to 1–3
            for dim in ("clarity", "accuracy", "retrievability"):
                if dim not in parsed:
                    raise ValueError(f"Missing dimension: {dim}")
                parsed[dim] = max(1, min(3, int(parsed[dim])))

            return parsed

        except Exception as e:
            print(f"        [Attempt {attempt}/{MAX_RETRIES}] Scoring error: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_WAIT)
            else:
                print(f"        All retries exhausted — pair will be rejected.")
                return {}


def stage2_filter(pairs: dict, client: OpenAI, min_score: int) -> tuple:
    """
    Score each QA pair with DeepSeek and filter by minimum score threshold.

    A pair passes if ALL three dimensions (clarity, accuracy, retrievability)
    have a score >= min_score.

    Args:
        pairs:     Dict of pairs that passed Stage 1.
        client:    Initialised OpenAI client pointing to DeepSeek.
        min_score: Minimum acceptable score for each dimension.

    Returns:
        (passed, rejected) — two dicts with the same structure as input pairs.
        Each entry has extra score fields: clarity, accuracy, retrievability, reason.
        Rejected entries also have 'reject_reason' and 'filter_stage' keys.
    """
    passed   = {}
    rejected = {}

    for qid, pair in tqdm(pairs.items(), desc="Stage 2 scoring"):
        scores = score_pair(client, pair["question"], pair["answer"])
        time.sleep(SLEEP_BETWEEN_CALLS)

        if not scores:
            # API failure — reject conservatively
            rejected[qid] = {
                **pair,
                "reject_reason": "API scoring failed after all retries",
                "filter_stage":  2,
            }
            continue

        # Attach scores to the pair for the report
        scored_pair = {
            **pair,
            "clarity":        scores["clarity"],
            "accuracy":       scores["accuracy"],
            "retrievability": scores["retrievability"],
            "score_reason":   scores.get("reason", ""),
        }

        # Find the lowest-scoring dimension
        lowest_dim   = min(
            ("clarity", "accuracy", "retrievability"),
            key=lambda d: scores[d]
        )
        lowest_score = scores[lowest_dim]

        if lowest_score < min_score:
            rejected[qid] = {
                **scored_pair,
                "reject_reason": (
                    f"{lowest_dim} score {lowest_score} < threshold {min_score}: "
                    f"{scores.get('reason', '')}"
                ),
                "filter_stage": 2,
            }
        else:
            passed[qid] = scored_pair

    return passed, rejected


# =============================================================================
# LOAD CHECKPOINT (resume support)
# =============================================================================

def load_stage2_checkpoint(output_dir: str) -> tuple:
    """
    Load any previously scored pairs to resume Stage 2 if interrupted.

    Returns:
        (stage2_passed, stage2_rejected) dicts, both empty if no checkpoint.
    """
    path = os.path.join(output_dir, "filter_stage2_checkpoint.json")
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        passed   = data.get("passed",   {})
        rejected = data.get("rejected", {})
        total = len(passed) + len(rejected)
        print(f"  Resumed Stage 2: {total} pairs already scored "
              f"({len(passed)} passed, {len(rejected)} rejected).")
        return passed, rejected
    return {}, {}


def save_stage2_checkpoint(output_dir: str, passed: dict, rejected: dict):
    """Save Stage 2 progress to disk."""
    path = os.path.join(output_dir, "filter_stage2_checkpoint.json")
    with open(path, 'w', encoding='utf-8') as f:
        json.dump({"passed": passed, "rejected": rejected}, f,
                  indent=2, ensure_ascii=False)


# =============================================================================
# SAVE FILTERED OUTPUTS
# =============================================================================

def save_filtered_outputs(output_dir: str,
                           final_pairs: dict,
                           all_rejected: dict):
    """
    Write filtered output files and a full audit report.

    Args:
        output_dir:   Where to write files.
        final_pairs:  Pairs that passed both stages.
        all_rejected: All rejected pairs from both stages combined.
    """
    # ── queries_filtered.json ─────────────────────────────────────────────────
    queries = {
        qid: {
            "query":  v["question"],
            "type":   v["query_type"],
            "source": v["source_type"],
        }
        for qid, v in final_pairs.items()
    }
    with open(os.path.join(output_dir, "queries_filtered.json"), 'w', encoding='utf-8') as f:
        json.dump(queries, f, indent=2, ensure_ascii=False)

    # ── answers_filtered.json ─────────────────────────────────────────────────
    answers = {qid: v["answer"] for qid, v in final_pairs.items()}
    with open(os.path.join(output_dir, "answers_filtered.json"), 'w', encoding='utf-8') as f:
        json.dump(answers, f, indent=2, ensure_ascii=False)

    # ── qrels_filtered.json ───────────────────────────────────────────────────
    qrels = {
        qid: {
            "doc_id":     v["paper_id"],
            "section_id": v["section_id"],
        }
        for qid, v in final_pairs.items()
    }
    with open(os.path.join(output_dir, "qrels_filtered.json"), 'w', encoding='utf-8') as f:
        json.dump(qrels, f, indent=2, ensure_ascii=False)

    # ── benchmark_filtered.csv ────────────────────────────────────────────────
    csv_path  = os.path.join(output_dir, "benchmark_filtered.csv")
    fieldnames = ["query_id", "question", "answer", "query_type", "source_type",
                  "paper_id", "section_id", "clarity", "accuracy", "retrievability"]
    with open(csv_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for qid, v in final_pairs.items():
            writer.writerow({"query_id": qid, **v})

    # ── filter_report.json ────────────────────────────────────────────────────
    report = {
        "summary": {
            "total_input":          len(final_pairs) + len(all_rejected),
            "total_passed":         len(final_pairs),
            "total_rejected":       len(all_rejected),
            "rejected_stage1":      sum(1 for v in all_rejected.values() if v.get("filter_stage") == 1),
            "rejected_stage2":      sum(1 for v in all_rejected.values() if v.get("filter_stage") == 2),
            "retention_rate":       f"{100 * len(final_pairs) / (len(final_pairs) + len(all_rejected)):.1f}%",
        },
        "rejected_pairs": {
            qid: {
                "question":     v.get("question", ""),
                "reject_reason": v.get("reject_reason", ""),
                "filter_stage": v.get("filter_stage", ""),
            }
            for qid, v in all_rejected.items()
        },
    }
    report_path = os.path.join(output_dir, "filter_report.json")
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print(f"\n  queries_filtered.json  →  {os.path.join(output_dir, 'queries_filtered.json')}")
    print(f"  answers_filtered.json  →  {os.path.join(output_dir, 'answers_filtered.json')}")
    print(f"  qrels_filtered.json    →  {os.path.join(output_dir, 'qrels_filtered.json')}")
    print(f"  benchmark_filtered.csv →  {csv_path}")
    print(f"  filter_report.json     →  {report_path}")


# =============================================================================
# MAIN
# =============================================================================

def filter_benchmark(input_dir: str, output_dir: str) -> None:
    """
    Full two-stage filtering pipeline.

    Loads benchmark_checkpoint.json (the raw generated pairs from
    generate_benchmark.py) and applies Stage 1 then Stage 2 filtering.
    """
    os.makedirs(output_dir, exist_ok=True)

    # ── Load raw generated pairs ──────────────────────────────────────────────
    checkpoint_path = os.path.join(input_dir, "benchmark_checkpoint.json")
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(
            f"Could not find benchmark_checkpoint.json at {checkpoint_path}.\n"
            f"Please run generate_benchmark.py first."
        )
    with open(checkpoint_path, 'r', encoding='utf-8') as f:
        raw_pairs = json.load(f)

    print(f"\n{'='*55}")
    print(f"  QA BENCHMARK QUALITY FILTER")
    print(f"{'='*55}")
    print(f"  Loaded {len(raw_pairs)} raw QA pairs.")

    # ── Stage 1: Rule-based filter ────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STAGE 1: Rule-based filter")
    print(f"{'='*55}\n")

    s1_passed, s1_rejected = stage1_filter(raw_pairs)

    print(f"  Input:    {len(raw_pairs)}")
    print(f"  Passed:   {len(s1_passed)}")
    print(f"  Rejected: {len(s1_rejected)}")

    if s1_rejected:
        # Show breakdown of rejection reasons
        reason_counts = {}
        for v in s1_rejected.values():
            # Normalise reason to category
            r = v.get("reject_reason", "unknown")
            if "short" in r or "long" in r:
                cat = "length violation"
            elif "non-retrievable" in r:
                cat = "non-retrievable phrase"
            elif "refusal" in r:
                cat = "model refusal in answer"
            elif "duplicate" in r:
                cat = "near-duplicate question"
            else:
                cat = "other"
            reason_counts[cat] = reason_counts.get(cat, 0) + 1
        print(f"\n  Rejection breakdown:")
        for cat, count in sorted(reason_counts.items(), key=lambda x: -x[1]):
            print(f"    {cat:<30} {count}")

    # ── Stage 2: LLM-based filter ─────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  STAGE 2: LLM-based filter  (min score: {LLM_MIN_SCORE}/3 per dimension)")
    print(f"{'='*55}\n")

    client = OpenAI(
        api_key  = DEEPSEEK_API_KEY,
        base_url = "https://api.deepseek.com",
    )

    # Resume support: load already-scored pairs
    s2_passed, s2_rejected = load_stage2_checkpoint(output_dir)
    already_scored = set(s2_passed.keys()) | set(s2_rejected.keys())

    # Only score pairs not yet processed
    remaining = {qid: v for qid, v in s1_passed.items() if qid not in already_scored}
    print(f"  Pairs to score: {len(remaining)} "
          f"({len(already_scored)} already done from checkpoint)\n")

    for qid, pair in tqdm(remaining.items(), desc="Scoring"):
        scores = score_pair(client, pair["question"], pair["answer"])
        time.sleep(SLEEP_BETWEEN_CALLS)

        if not scores:
            s2_rejected[qid] = {
                **pair,
                "reject_reason": "API scoring failed after all retries",
                "filter_stage":  2,
            }
            continue

        scored_pair = {
            **pair,
            "clarity":        scores["clarity"],
            "accuracy":       scores["accuracy"],
            "retrievability": scores["retrievability"],
            "score_reason":   scores.get("reason", ""),
        }

        lowest_dim   = min(
            ("clarity", "accuracy", "retrievability"),
            key=lambda d: scores[d]
        )
        lowest_score = scores[lowest_dim]

        if lowest_score < LLM_MIN_SCORE:
            s2_rejected[qid] = {
                **scored_pair,
                "reject_reason": (
                    f"{lowest_dim} score {lowest_score} < {LLM_MIN_SCORE}: "
                    f"{scores.get('reason', '')}"
                ),
                "filter_stage": 2,
            }
        else:
            s2_passed[qid] = scored_pair

        # Checkpoint every 50 pairs
        if (len(s2_passed) + len(s2_rejected)) % 50 == 0:
            save_stage2_checkpoint(output_dir, s2_passed, s2_rejected)

    # Final checkpoint
    save_stage2_checkpoint(output_dir, s2_passed, s2_rejected)

    print(f"\n  Input:    {len(s1_passed)}")
    print(f"  Passed:   {len(s2_passed)}")
    print(f"  Rejected: {len(s2_rejected)}")

    # ── Save outputs ──────────────────────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  Saving filtered outputs")
    print(f"{'='*55}")

    all_rejected = {**s1_rejected, **s2_rejected}
    save_filtered_outputs(output_dir, s2_passed, all_rejected)

    # ── Final summary ─────────────────────────────────────────────────────────
    total_in  = len(raw_pairs)
    total_out = len(s2_passed)

    # Score distribution for passed pairs
    score_dims = ("clarity", "accuracy", "retrievability")
    avg_scores = {}
    for dim in score_dims:
        scores_list = [v[dim] for v in s2_passed.values() if dim in v]
        avg_scores[dim] = sum(scores_list) / len(scores_list) if scores_list else 0

    print(f"\n{'='*55}")
    print(f"  FILTERING COMPLETE")
    print(f"{'='*55}")
    print(f"  Input pairs:           {total_in}")
    print(f"  Rejected (Stage 1):    {len(s1_rejected)}")
    print(f"  Rejected (Stage 2):    {len(s2_rejected)}")
    print(f"  Final pairs retained:  {total_out}")
    print(f"  Retention rate:        {100 * total_out / total_in:.1f}%")
    if avg_scores:
        print(f"\n  Avg scores (passed pairs):")
        for dim, avg in avg_scores.items():
            print(f"    {dim:<16} {avg:.2f} / 3.00")
    print(f"{'='*55}\n")


# =============================================================================
# RUN
# =============================================================================

filter_benchmark(
    input_dir  = INPUT_DIR,
    output_dir = OUTPUT_DIR,
)


  QA BENCHMARK QUALITY FILTER
  Loaded 1000 raw QA pairs.

  STAGE 1: Rule-based filter

  Input:    1000
  Passed:   959
  Rejected: 41

  Rejection breakdown:
    non-retrievable phrase         26
    length violation               10
    near-duplicate question        5

  STAGE 2: LLM-based filter  (min score: 2/3 per dimension)

  Resumed Stage 2: 959 pairs already scored (821 passed, 138 rejected).
  Pairs to score: 0 (959 already done from checkpoint)



Scoring: 0it [00:00, ?it/s]



  Input:    959
  Passed:   821
  Rejected: 138

  Saving filtered outputs

  queries_filtered.json  →  /content/drive/MyDrive/arxiv_data/corpus_output/queries_filtered.json
  answers_filtered.json  →  /content/drive/MyDrive/arxiv_data/corpus_output/answers_filtered.json
  qrels_filtered.json    →  /content/drive/MyDrive/arxiv_data/corpus_output/qrels_filtered.json
  benchmark_filtered.csv →  /content/drive/MyDrive/arxiv_data/corpus_output/benchmark_filtered.csv
  filter_report.json     →  /content/drive/MyDrive/arxiv_data/corpus_output/filter_report.json

  FILTERING COMPLETE
  Input pairs:           1000
  Rejected (Stage 1):    41
  Rejected (Stage 2):    138
  Final pairs retained:  821
  Retention rate:        82.1%

  Avg scores (passed pairs):
    clarity          2.87 / 3.00
    accuracy         2.90 / 3.00
    retrievability   2.53 / 3.00

